# Cabardo Studio AI Generator — Free Colab T4 16 GB Backend (Zero-OOM Two-Phase)

**Engineered specifically for Free Google Colab T4 (`12.67 GB` RAM / `14.6 GB` VRAM — Zero Training · Zero OOM Crashes):**
- **Two-Phase GPU-Only Execution:** Encodes prompts with 4-bit NF4 T5-XXL directly on GPU (~2.7 GB VRAM), unloads text encoder completely, then runs the video diffusion transformer + VAE directly on GPU — keeping System RAM under **~3.5 GB / 12.67 GB** at all times!
- **`THUDM/CogVideoX-2b`** (Default T2V · 13.7 GB download · Native T4 FP16 Tensor Cores)
- **`Lightricks/LTX-Video-0.9.5`** (2B DiT · 24 fps **Text-to-Video**, **Image-to-Video**, and **First→Last Frame**)
- **`cvssp/audioldm2` + FFmpeg AAC Muxing**: Synchronized AI foley & soundscape

**How to run:**
1. Confirm top-right says **T4** (if not: `Runtime -> Change runtime type -> T4 GPU -> Save`).
2. Click **Runtime -> Run all** (~60s setup).
3. Scroll below **Step 2** and click the bright green **`Click Here to Open Cabardo Studio (Auto-Connected)`** button.


In [ ]:
# @title Step 1: Verify Free NVIDIA T4 16 GB GPU & Install Dependencies (~40s)
import os
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU attached! Click Runtime -> Change runtime type -> T4 GPU -> Save, then click Run All."
    )

gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
print(f"GPU Ready: {gpu_name} ({vram_gb:.1f} GB VRAM)", flush=True)
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv
!pip -q install -U "diffusers>=0.35.0" transformers accelerate bitsandbytes safetensors sentencepiece protobuf ftfy "imageio[ffmpeg]" pillow numpy scipy fastapi uvicorn
print("All Cabardo Studio Colab T4 dependencies installed cleanly.", flush=True)


In [ ]:
# @title Step 2: Launch Zero-OOM Colab T4 Video + Audio Server (FastAPI + 1-Click Auto-Connect)
import asyncio
import base64
import gc
import io
import os
import re
import shutil
import subprocess
import threading
import time
import urllib.request
import uuid
import wave
from typing import Optional

import numpy as np
import torch
from PIL import Image, ImageOps
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import FileResponse
from pydantic import BaseModel
import uvicorn

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU attached! Click Runtime -> Change runtime type -> T4 GPU -> Save, then Run All again."
    )

os.makedirs("outputs", exist_ok=True)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

def _total_ram_gb() -> float:
    try:
        with open("/proc/meminfo", "r", encoding="utf-8") as f:
            for line in f:
                if line.startswith("MemTotal:"):
                    return int(line.split()[1]) / (1024 ** 2)
    except Exception:
        pass
    return 12.6

TOTAL_RAM_GB = _total_ram_gb()
FREE_COLAB_LOW_RAM = TOTAL_RAM_GB < 18.0
COMPACT_T5_REPO = "THUDM/CogVideoX-2b"

app = FastAPI(title="Cabardo Studio Zero-OOM Colab T4 Backend")
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=False,
    allow_methods=["*"],
    allow_headers=["*"],
)

GPU_LOCK = threading.Lock()
STATE = {
    "active_pipe_key": None,
    "pipe": None,
    "jobs": {},
}

def _free_gpu():
    if STATE["pipe"] is not None:
        del STATE["pipe"]
        STATE["pipe"] = None
        STATE["active_pipe_key"] = None
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def _decode_pil(data_url: str, width: int, height: int) -> Image.Image:
    raw_b64 = data_url.split(",", 1)[1]
    img = Image.open(io.BytesIO(base64.b64decode(raw_b64))).convert("RGB")
    return ImageOps.fit(img, (width, height), method=Image.Resampling.LANCZOS)

def _encode_t5_4bit(prompt: str, negative_prompt: str, max_length: int, out_dtype: torch.dtype, need_mask: bool):
    """Phase 1: Load 4-bit NF4 T5-XXL directly on GPU (~2.7GB VRAM, ~1.5GB RAM), encode, then immediately free."""
    _free_gpu()
    from transformers import BitsAndBytesConfig, T5EncoderModel, T5Tokenizer
    bnb_cfg = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
    )
    tokenizer = T5Tokenizer.from_pretrained(COMPACT_T5_REPO, subfolder="tokenizer")
    text_encoder = T5EncoderModel.from_pretrained(
        COMPACT_T5_REPO,
        subfolder="text_encoder",
        quantization_config=bnb_cfg,
        device_map={"": 0},
        low_cpu_mem_usage=True,
    )
    try:
        def _enc(text: str):
            inp = tokenizer(
                text,
                padding="max_length",
                max_length=max_length,
                truncation=True,
                add_special_tokens=True,
                return_tensors="pt",
            )
            ids = inp.input_ids.to("cuda")
            mask = inp.attention_mask.to("cuda")
            with torch.no_grad():
                if need_mask:
                    embeds = text_encoder(ids, attention_mask=mask)[0]
                else:
                    embeds = text_encoder(ids)[0]
            return embeds.to(dtype=out_dtype, device="cpu"), mask.bool().to("cpu")

        p_emb, p_mask = _enc(prompt)
        n_emb, n_mask = _enc(negative_prompt or "")
        return p_emb, p_mask, n_emb, n_mask
    finally:
        del text_encoder
        del tokenizer
        gc.collect()
        torch.cuda.empty_cache()

def _load_image_pipe(model_key: str):
    target_key = f"img:{model_key}"
    if STATE["active_pipe_key"] == target_key and STATE["pipe"] is not None:
        return STATE["pipe"]
    _free_gpu()
    from diffusers import AutoPipelineForText2Image
    if model_key == "sdxl-turbo":
        pipe = AutoPipelineForText2Image.from_pretrained(
            "stabilityai/sdxl-turbo", torch_dtype=torch.float16, variant="fp16", low_cpu_mem_usage=True
        ).to("cuda")
    else:
        pipe = AutoPipelineForText2Image.from_pretrained(
            "stabilityai/stable-diffusion-xl-base-1.0",
            torch_dtype=torch.float16,
            variant="fp16",
            use_safetensors=True,
            low_cpu_mem_usage=True,
        ).to("cuda")
        pipe.enable_vae_slicing()
        pipe.enable_vae_tiling()
    STATE["pipe"] = pipe
    STATE["active_pipe_key"] = target_key
    return pipe

def _load_video_phase2_pipe(model_key: str, mode_variant: str):
    """Phase 2: Load ONLY transformer + VAE directly onto GPU (text_encoder=None). Zero CPU RAM offload!"""
    target_key = f"vid2:{model_key}:{mode_variant}"
    if STATE["active_pipe_key"] == target_key and STATE["pipe"] is not None:
        return STATE["pipe"]
    _free_gpu()
    if model_key == "ltx-video":
        repo = "Lightricks/LTX-Video-0.9.5"
        ltx_dtype = torch.bfloat16
        if mode_variant == "flf":
            from diffusers import LTXConditionPipeline
            pipe = LTXConditionPipeline.from_pretrained(
                repo, text_encoder=None, tokenizer=None, torch_dtype=ltx_dtype, low_cpu_mem_usage=True
            ).to("cuda")
        elif mode_variant == "i2v":
            from diffusers import LTXImageToVideoPipeline
            pipe = LTXImageToVideoPipeline.from_pretrained(
                repo, text_encoder=None, tokenizer=None, torch_dtype=ltx_dtype, low_cpu_mem_usage=True
            ).to("cuda")
        else:
            from diffusers import LTXPipeline
            pipe = LTXPipeline.from_pretrained(
                repo, text_encoder=None, tokenizer=None, torch_dtype=ltx_dtype, low_cpu_mem_usage=True
            ).to("cuda")
        pipe.vae.enable_tiling()
    else:
        from diffusers import CogVideoXPipeline
        pipe = CogVideoXPipeline.from_pretrained(
            "THUDM/CogVideoX-2b",
            text_encoder=None,
            tokenizer=None,
            torch_dtype=torch.float16,
            low_cpu_mem_usage=True,
        ).to("cuda")
        pipe.vae.enable_slicing()
        pipe.vae.enable_tiling()
    STATE["pipe"] = pipe
    STATE["active_pipe_key"] = target_key
    return pipe

def _synthesize_and_mux_audio(video_path: str, prompt: str, duration_sec: float, seed: int) -> bool:
    if not shutil.which("ffmpeg"):
        return False
    try:
        _free_gpu()
        from diffusers import AudioLDM2Pipeline
        ap = AudioLDM2Pipeline.from_pretrained(
            "cvssp/audioldm2", torch_dtype=torch.float16, low_cpu_mem_usage=True
        ).to("cuda")
        try:
            gen = torch.Generator(device="cuda").manual_seed(seed)
            audio_len = max(2.5, min(8.0, float(duration_sec)))
            wav = ap(
                prompt=f"Realistic synchronized cinematic foley sound effects and atmosphere: {prompt}",
                negative_prompt="low quality, harsh noise, distortion, clipping",
                num_inference_steps=20,
                audio_length_in_s=audio_len,
                generator=gen,
            ).audios[0]
        finally:
            del ap
            gc.collect()
            torch.cuda.empty_cache()

        wav_path = video_path + ".wav"
        arr = np.asarray(wav, dtype=np.float32).reshape(-1)
        peak = float(np.max(np.abs(arr))) if arr.size else 1.0
        if peak > 1.0:
            arr = arr / peak
        pcm = np.clip(arr * 32767.0, -32768, 32767).astype(np.int16)
        with wave.open(wav_path, "wb") as wf:
            wf.setnchannels(1)
            wf.setsampwidth(2)
            wf.setframerate(16000)
            wf.writeframes(pcm.tobytes())

        muxed_path = video_path + ".mux.mp4"
        proc = subprocess.run(
            [
                "ffmpeg", "-y", "-hide_banner", "-loglevel", "error",
                "-i", video_path, "-i", wav_path,
                "-map", "0:v:0", "-map", "1:a:0",
                "-c:v", "copy", "-c:a", "aac", "-b:a", "192k",
                "-shortest", "-movflags", "+faststart", muxed_path,
            ],
            capture_output=True,
            text=True,
            timeout=60,
        )
        if proc.returncode == 0 and os.path.isfile(muxed_path):
            os.replace(muxed_path, video_path)
            os.remove(wav_path)
            return True
    except Exception as exc:
        print("Audio synthesis skipped:", exc, flush=True)
    return False

class ImageRequest(BaseModel):
    prompt: str
    negative_prompt: str = "blurry, low quality, distorted, watermark"
    width: int = 1024
    height: int = 576
    steps: int = 20
    cfg_scale: float = 7.0
    seed: int = -1
    model: str = "sdxl-turbo"

class VideoRequest(BaseModel):
    prompt: str
    negative_prompt: str = "blurry, jittery, distorted, worst quality"
    width: int = 704
    height: int = 480
    duration_sec: int = 3
    steps: int = 24
    cfg_scale: float = 6.0
    seed: int = -1
    model: str = "cogvideox-2b"
    mode: str = "t2v"
    start_frame_data_url: Optional[str] = None
    end_frame_data_url: Optional[str] = None
    generate_audio: bool = True

@app.get("/api/health")
def health():
    gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
    total_gb = round(torch.cuda.get_device_properties(0).total_memory / (1024 ** 3), 1) if torch.cuda.is_available() else 0.0
    free_gb = round(torch.cuda.mem_get_info(0)[0] / (1024 ** 3), 1) if torch.cuda.is_available() else 0.0
    return {
        "ok": True,
        "gpu_name": gpu_name,
        "vram_total_gb": total_gb,
        "vram_free_gb": free_gb,
        "ram_total_gb": round(TOTAL_RAM_GB, 1),
        "active_pipeline": STATE["active_pipe_key"],
        "models": ["sdxl-turbo", "sdxl-base", "cogvideox-2b", "ltx-video"],
    }

@app.post("/api/image")
def generate_image(req: ImageRequest):
    actual_seed = int(time.time() * 1000) % (2**31) if req.seed < 0 else int(req.seed)
    with GPU_LOCK:
        pipe = _load_image_pipe(req.model)
        is_turbo = req.model == "sdxl-turbo"
        w = min(max(256, (req.width // 16) * 16), 768 if is_turbo else 1024)
        h = min(max(256, (req.height // 16) * 16), 768 if is_turbo else 1024)
        steps = min(max(1, req.steps), 4) if is_turbo else min(max(12, req.steps), 35)
        cfg = 0.0 if is_turbo else float(req.cfg_scale)
        gen = torch.Generator(device="cuda").manual_seed(actual_seed)
        img = pipe(
            prompt=req.prompt,
            negative_prompt=None if is_turbo else req.negative_prompt,
            width=w,
            height=h,
            num_inference_steps=steps,
            guidance_scale=cfg,
            generator=gen,
        ).images[0]
        fname = f"img_{int(time.time())}_{actual_seed}.png"
        fpath = os.path.join("outputs", fname)
        img.save(fpath)
        buf = io.BytesIO()
        img.save(buf, format="PNG")
        b64 = base64.b64encode(buf.getvalue()).decode("ascii")
        return {
            "ok": True,
            "seed": actual_seed,
            "width": w,
            "height": h,
            "output_path": f"/outputs/{fname}",
            "data_url": f"data:image/png;base64,{b64}",
        }

def _run_video_worker(job_id: str, req: VideoRequest):
    from diffusers.utils import export_to_video
    job = STATE["jobs"][job_id]
    try:
        actual_seed = int(time.time() * 1000) % (2**31) if req.seed < 0 else int(req.seed)
        has_start = bool(req.start_frame_data_url and req.start_frame_data_url.startswith("data:image/"))
        has_end = bool(req.end_frame_data_url and req.end_frame_data_url.startswith("data:image/"))
        mode_variant = "flf" if (has_start and has_end) else ("i2v" if (req.mode == "i2v" and has_start) else "t2v")

        # Guard Free Colab T4 12.67GB RAM: never load 47GB 13B models; route to Zero-OOM 2B models
        if mode_variant in ("i2v", "flf") or req.model == "ltx-video":
            model_key = "ltx-video"
        else:
            model_key = "cogvideox-2b"

        job["status"] = "processing"
        job["progress"] = 12
        job["phase"] = f"Phase 1/3: Encoding prompt in 4-bit NF4 on T4 GPU ({model_key})..."
        print(f"[Job {job_id}] Starting {model_key} ({mode_variant}) on T4 (RAM={TOTAL_RAM_GB:.1f}GB)", flush=True)

        with GPU_LOCK:
            is_ltx = model_key == "ltx-video"
            p_emb, p_mask, n_emb, n_mask = _encode_t5_4bit(
                req.prompt,
                req.negative_prompt,
                max_length=128 if is_ltx else 226,
                out_dtype=torch.bfloat16 if is_ltx else torch.float16,
                need_mask=is_ltx,
            )

            job["progress"] = 32
            job["phase"] = f"Phase 2/3: Loading {model_key} transformer on T4 GPU (Zero-OOM)..."
            pipe = _load_video_phase2_pipe(model_key, mode_variant)

            job["progress"] = 45
            job["phase"] = f"Phase 2/3: Diffusing AI video frames on T4 GPU ({model_key})..."
            gen = torch.Generator(device="cuda").manual_seed(actual_seed)

            if is_ltx:
                w = max(384, min(704, (req.width // 32) * 32))
                h = max(384, min(480, (req.height // 32) * 32))
                fps = 24
                num_frames = min(73, max(25, int(req.duration_sec * fps) + 1))
                num_frames = ((num_frames - 1) // 8) * 8 + 1
                kwargs = dict(
                    prompt_embeds=p_emb.to("cuda"),
                    prompt_attention_mask=p_mask.to("cuda"),
                    negative_prompt_embeds=n_emb.to("cuda"),
                    negative_prompt_attention_mask=n_mask.to("cuda"),
                    width=w,
                    height=h,
                    num_frames=num_frames,
                    num_inference_steps=min(max(20, req.steps), 35),
                    guidance_scale=3.5,
                    generator=gen,
                )
                if mode_variant == "flf" and req.start_frame_data_url and req.end_frame_data_url:
                    from diffusers.pipelines.ltx.pipeline_ltx_condition import LTXVideoCondition
                    kwargs["conditions"] = [
                        LTXVideoCondition(image=_decode_pil(req.start_frame_data_url, w, h), frame_index=0, strength=1.0),
                        LTXVideoCondition(image=_decode_pil(req.end_frame_data_url, w, h), frame_index=num_frames - 1, strength=1.0),
                    ]
                elif mode_variant == "i2v" and req.start_frame_data_url:
                    kwargs["image"] = _decode_pil(req.start_frame_data_url, w, h)
                frames = pipe(**kwargs).frames[0]
            else:
                fps = 8
                num_frames = min(49, max(17, int(req.duration_sec * fps) + 1))
                frames = pipe(
                    prompt_embeds=p_emb.to("cuda"),
                    negative_prompt_embeds=n_emb.to("cuda"),
                    num_videos_per_prompt=1,
                    num_inference_steps=min(max(20, req.steps), 35),
                    num_frames=num_frames,
                    guidance_scale=float(req.cfg_scale or 6.0),
                    generator=gen,
                ).frames[0]

            job["progress"] = 84
            job["phase"] = "Phase 3/3: Encoding H.264 MP4 clip..."
            fname = f"vid_{job_id}_{actual_seed}.mp4"
            fpath = os.path.join("outputs", fname)
            export_to_video(frames, fpath, fps=fps)

            has_audio = False
            if req.generate_audio:
                job["progress"] = 92
                job["phase"] = "Phase 3/3: Adding synchronized AI foley audio (AudioLDM2)..."
                has_audio = _synthesize_and_mux_audio(fpath, req.prompt, len(frames) / float(fps), actual_seed)

            job["status"] = "completed"
            job["progress"] = 100
            job["phase"] = f"Completed on Colab T4 ({model_key})"
            job["output_path"] = f"/outputs/{fname}"
            job["has_audio"] = has_audio
            print(f"[Job {job_id}] COMPLETED: {fpath} (audio={has_audio})", flush=True)
    except Exception as exc:
        import traceback
        traceback.print_exc()
        job["status"] = "failed"
        job["error"] = str(exc)

@app.post("/api/video")
def queue_video(req: VideoRequest):
    job_id = f"job_{uuid.uuid4().hex[:12]}"
    STATE["jobs"][job_id] = {
        "job_id": job_id,
        "status": "queued",
        "progress": 5,
        "phase": "Queued for NVIDIA T4 16 GB GPU...",
        "output_path": None,
        "has_audio": False,
        "error": None,
    }
    threading.Thread(target=_run_video_worker, args=(job_id, req), daemon=True).start()
    return {"ok": True, "job_id": job_id, "status": "queued"}

@app.get("/api/job/{job_id}")
def get_job(job_id: str):
    if not re.fullmatch(r"[A-Za-z0-9_-]{4,64}", job_id) or job_id not in STATE["jobs"]:
        raise HTTPException(status_code=404, detail="Job not found")
    return STATE["jobs"][job_id]

@app.get("/outputs/{filename}")
def get_output(filename: str):
    if not re.fullmatch(r"[A-Za-z0-9._-]{3,128}", filename):
        raise HTTPException(status_code=400, detail="Invalid filename")
    fpath = os.path.join("outputs", filename)
    if not os.path.isfile(fpath):
        raise HTTPException(status_code=404, detail="Output file not found")
    return FileResponse(fpath)

def _start_api():
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    config = uvicorn.Config(app, host="127.0.0.1", port=7860, log_level="warning")
    server = uvicorn.Server(config)
    loop.run_until_complete(server.serve())

if not globals().get("_FLOWSTUDIO_SERVER_STARTED"):
    threading.Thread(target=_start_api, daemon=True).start()
    _FLOWSTUDIO_SERVER_STARTED = True

api_ready = False
for _ in range(30):
    try:
        with urllib.request.urlopen("http://127.0.0.1:7860/api/health", timeout=2) as r:
            if r.status == 200:
                print("Colab T4 API health OK:", r.read().decode("utf-8"), flush=True)
                api_ready = True
                break
    except Exception:
        time.sleep(0.5)

if not api_ready:
    raise RuntimeError("Local FastAPI server failed to start on port 7860.")

cf_bin = "/usr/local/bin/cloudflared"
if not os.path.exists(cf_bin):
    print("Downloading official cloudflared binary...", flush=True)
    urllib.request.urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        cf_bin,
    )
    os.chmod(cf_bin, 0o755)

print("Starting free Cloudflare Quick Tunnel to your T4 GPU...", flush=True)
proc = subprocess.Popen(
    [cf_bin, "tunnel", "--url", "http://127.0.0.1:7860", "--no-autoupdate"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

tunnel_url = None
recent_lines = []
for line in iter(proc.stdout.readline, ""):
    recent_lines.append(line.strip())
    if len(recent_lines) > 20:
        recent_lines.pop(0)
    m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if m and not tunnel_url:
        tunnel_url = m.group(0)
        connect_link = f"https://flow-studio-pwa.vercel.app/?colab={tunnel_url}"
        try:
            from IPython.display import HTML, display
            display(HTML(
                f'<div style="background:#0d1f17;border:2px solid #29c58f;border-radius:14px;padding:18px;font-family:sans-serif;color:#fff;max-width:680px">'
                f'<div style="font-size:16px;font-weight:700;color:#29c58f;margin-bottom:8px">COLAB T4 GPU SERVER IS LIVE!</div>'
                f'<a href="{connect_link}" target="_blank" style="display:inline-block;background:#ffffff;color:#111;font-weight:700;padding:12px 20px;border-radius:999px;text-decoration:none;margin:6px 0 12px 0">Click Here to Open Cabardo Studio (Auto-Connected)</a>'
                f'<div style="font-size:12px;color:#cbd5e1;margin-bottom:4px">Or copy this Cloudflare URL manually:</div>'
                f'<code style="display:block;background:#000;color:#29c58f;padding:10px 12px;border-radius:8px;font-size:14px;user-select:all">{tunnel_url}</code>'
                f'<div style="font-size:11px;color:#94a3b8;margin-top:8px">Keep this Colab tab open while generating videos.</div>'
                f'</div>'
            ))
        except Exception:
            pass
        print("=" * 72, flush=True)
        print("COLAB T4 GPU SERVER IS LIVE!", flush=True)
        print("1-CLICK AUTO-CONNECT: " + connect_link, flush=True)
        print("CLOUDFLARE URL:       " + tunnel_url, flush=True)
        print("Keep this cell running while generating real AI video + audio.", flush=True)
        print("=" * 72, flush=True)

if not tunnel_url:
    raise RuntimeError("Cloudflare tunnel exited without printing a URL:\n" + "\n".join(recent_lines))
